<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 9: Convolutional Variational Autoencoders

#### Tim Moroney, 2026

A lesson where we introduce convolutional variational autoencoders as a stepping stone towards generative image models.

# Package management

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Probability concepts and notation

Autoencoders are built upon the idea of using probability as a measure of uncertainty: this is the _Bayesian_ interpretation of probability.  We'll need the following probability concepts throughout this lesson.

|  | |
| -------------| ------- |
| $p(x)$ | Probability density function (PDF) of $X$ |
| $p(x,z)$ | Joint PDF of $X$ and $Z$|
| $p(z) = \int p(x,z)\, \textrm{d} x $ | Marginal PDF of $Z$ obtained from joint|
| $p(z\vert x) = \Large \frac{p(x,z)}{p(x)} $ | Conditional PDF of $Z$ given $X=x$ |
| $p(x\vert z) = \Large\frac{p(z\vert x)\, p(x)}{p(z)} $ | Bayes' theorem |
| $\mathbb{E}[g(X)] = \int g(x) p(x)\, \textrm{d}x$ | Expected value of $g(X)$ where $X$ has PDF $p(x)$|
| $\mathbb{E}_{X \sim p} [g(X)] = \int g(x) p(x)\, \textrm{d}x$ | Expected value of $g(X)$ (more explicit notation) |
| $f(x) = (2\pi)^{-d/2}\det(\Sigma)^{-1/2}\exp\left(-\frac{1}{2}(x-\mu)^\top\Sigma^{-1}(x-\mu)\right)$ | Gaussian PDF in $\mathbb{R}^d$  |
| $f(x) = (2\pi\sigma^2)^{-d/2}\exp\left(-\frac{\Vert x - \mu\Vert^2}{2\sigma^2}\right)$ | Isotropic Gaussian PDF in $\mathbb{R}^d$ $\left(\Sigma = \sigma^2\textrm{I}_d \right)$  |


# Image space and latent space

The space of all possible images is incomprehensibly vast.  Even considering only $32 \times 32$ greyscale images, there are $256^{32^2} \approx 10^{2500}$ possible images, which is crazy.  But nearly all of these possible images are not images _of_ anything -- they're just noise. Within the space of all possible images, the actual images of "something" occupy a much, much, much lower dimensional _manifold_.  It's impossible to visualise this in high-dimensional space, so here's a low-dimensional analogy: a squiggly curve in 3D space.

In our little analogy, $\mathcal{X} = \mathbb{R}^3$ represents the space of all possible images.  So every 3D coordinate $x = (x_1, x_2, x_3)$ is a different image.  But most of them are just noise.  The cyan dot, for example, represents an image that, like most others, is just noise.  The squiggly curve represents the lower-dimensional manifold on which all the genuine images of something can be found.  The dark blue and red dots are both genuine images in this analogy, since they're on the manifold.

In [ ]:
z = range(0, 3, length=1000)
x1 = 0.1*sin.(15*z)
x2 = cos.(z)+0.2*cos.(20*z)
x3 = sin.(z)+0.2*sin.(20*z)
f = Figure()
ax3 = f[1,1] = Axis3(f, xlabelsize=20, ylabelsize=20, zlabelsize=20, xlabel=L"x_1", ylabel=L"x_2", zlabel=L"x_3")
lines!(ax3, x1, x2, x3, color=z, linewidth=10)
idx = 742
scatter!(x1[idx], x2[idx], x3[idx], color=:red, markersize=20)
idx = 890
scatter!(x1[idx], x2[idx], x3[idx], color=:blue, markersize=20)
scatter!(0, 0.9, 1, color=:cyan, markersize=20)
f

#

Because the manifold is lower dimensional, any point on it can be described with fewer than 3 coordinates.  In our example, it's just a made-up squiggly curve parameterised by a single variable $z \in [0,3]$.  So every point on the manifold can be described using a single coordinate: the value of $z$.

So there is a lower-dimensional **latent** representation of all images on the manifold.  The true image space is $\mathcal{X} = \mathbb{R}^3$ consisting of $(x_1,x_2,x_3)$ coordinates, and the _latent space_ is one-dimensional $\mathcal{Z} = \mathbb{R}$, consisting of the $z$ coordinate.

# Autoencoder

The job of an **autoencoder** is to exploit this redundancy in real data(whether they be images or other kinds of inputs), to find a compressed, lower-dimensional representation that nonetheless can be used to reconstruct the original.  The autoencoder actually comprises two components: the encoder and the decoder.

We will use the notation $x \in \mathcal{X}$ as the input to the **encoder**.  Although we have images in mind for our application, the presentation will assume the input space $\mathcal{X} = \mathbb{R}^{d_1}$.  We gain nothing by complicating things any further -- this is a situation where just conceptually vectorising an image into a big column works fine.  To keep things concrete in the discussion, we will continue to refer to $x$ as an "image" and $\mathcal{X}$ as "image space".  Likewise we will refer to $z$ as a **latent variable** and $\mathcal{Z}$ as "latent space".

It is the job of the **encoder** to map $x \in \mathcal{X}$ to a latent variable $z \in \mathcal{Z}$, where the latent space is $\mathcal{Z} = \mathbb{R}^{d_2}$ for some $d_2 < d_1$.  And it is the job of the **decoder** to map a latent variable $z \in \mathcal{Z}$ back to some $x \in \mathcal{X}$.  

Ideally, you might imagine that the sole requirement of the encoder/decoder pair should be that the round-trip from an image to latent space and back again recovers the original image, i.e.
$$
x \approx \textrm{decode}(\textrm{encode}(x))
$$
for all genuine images (i.e. not just noise) $x$.

## Limitations
This is indeed how a standard autoencoder would be trained: to minimise the (squared) reconstruction error $$\|x - \textrm{decode}(\textrm{encode}(x))\|^2\,.$$ But this turns out to not quite be satisfactory in practice.  What one finds when training with a loss function based just on reconstruction error is that there is no structure to the learned latent space.  Yes, for any $x$ _that it was trained on_, it will encode to some $z$, which will reliably decode back to $x$.  But there is no guarantee that e.g. pictures of cats will all encode to similar latent representations.  Or that a small change to a latent variable $z$ will result in a small change to the decoded $x$.  And there is certainly no reason to expect that any arbitrary $z$ (not from the training set) will decode to anything sensible -- it will likely just decode to noise.

## Regularisation

What we really need is a form of **regularisation** in the loss function.  A term that, somehow, enforces _smoothness_ and _connectedness_ of the latent space.  So that nearby $z$s decode to nearby $x$s.  And so that interpolating between two latent variables $z_1$ and $z_2$ always decodes to a sensible image.

The great breakthrough made in **_variational_ autoencoders** was to realise that the missing piece of the puzzle is to acknowledge the _uncertainty_ in the encoding and decoding process.

Return to our made-up example.  Notice how the manifold comes close to crossing through itself in certain places.  For example, consider the red dot.  Which value of $z$ does it correspond to?  As it happens, both $z = 1.77$ and $z = 2.23$ seem very consistent with the location of this dot.  We'll mark these two points with white and black crosses.

In [ ]:
@show z[590]
@show z[742]
scatter!(x1[590], x2[590], x3[590], marker=:x, color=:white)
scatter!(x1[742], x2[742], x3[742], marker=:x, color=:black)
f

##
So even with ideal mathematical formulas, there are challenges with mapping from $\mathcal{X}$ to $\mathcal{Z}$. If we were trying to learn the encoding from data only, we would have to admit to being very _uncertain_ about which value of $z$ the red dot actually corresponds to.  Perhaps at best we'd be able to say that it's somewhere in or around the range $1.7 \leq z_\textrm{red} \leq 2.3$.

Conversely, the blue dot corresponds to $z_\textrm{blue} \approx 2.7$ with almost no uncertainty.

In [ ]:
@show z[890]
scatter!(x1[890], x2[890], x3[890], marker=:x, color=:white)
f

# Encoder and Decoder

So to build a robust encoder/decoder pair, we need a mechanism for the encoder to quantify not just its predicted latent $z$, but also how _uncertain_ it is about that prediction.  This is how probability enters the story: as a measure of uncertainty.  With this clever improvement, we will now aim to build an encoder in the form of a probability density function for the latent $z$ _given_ the image $x$:
$$
\textrm{encoder}: q_\phi(z|x)
$$
where $\phi$ denotes the parameters to be learned.  Similarly, the decoder will be represented by a probability density function for the image $x$ _given_ the latent $z$:
$$
\textrm{decoder}: p_\theta(x|z)
$$
where $\theta$ is a separate set of parameters to be learned for the decoder.

# Maximum likelihood objective
This gives us a new way to formulate the learning problem: as a **maximum likelihood** optimisation.  Let $p_\textrm{data}$ denote the probability density function of real images.  We can't observe $p_\textrm{data}$ directly, but we assume that our dataset comprises samples that are drawn from it.

Our goal is to construct a model $p_\theta$ that approximates the true distribution of images $p_\textrm{data}$.    The maximum likelihood objective is thus

$$
\max_\theta\, \mathbb{E}_{x\sim p_\textrm{data}} [\log p_\theta(x)]\,.
$$

We estimate this expected value by drawing $N$ sample images $x_i$ from the distribution $p_\textrm{data}$ and computing the sample mean:

$$
S(\theta) = \frac{1}{N} \sum_{i=1}^N \log p_\theta(x_i) \approx \mathbb{E}_{x\sim p_\textrm{data}} [\log p_\theta(x)]
$$

and seek the parameters $\theta$ satisfying

$$
\theta = \textrm{argmax}_{\theta^*} S(\theta^*).
$$

This is a good start, but there is a lot of maths between here and the finish line!  Let's get to it.

[Side note: the log in this formulation is _not_ optional, and _not_ merely a trick to simplify calculations.  Taking the log is fundamental to maximum likelihood optimisation -- see the exercises to confirm why!]

We can express $p_\theta$ in terms of all possible latents $z$ that $x$ might have encoded to:
$$
p_\theta(x) = \int p_\theta(x,z)\, \textrm{d}z
$$
where $p_\theta(x,z)$ is the _joint distribution_ of images $x$ and their latents $z$.
There's no hope to ever actually compute this integral over the full latent space $\mathcal{Z}$, and anyway we need to somehow introduce our _encoding_ distribution $q_\phi(z|x)$.  The statisticians have a trick for this: [Monte-Carlo](https://en.wikipedia.org/wiki/Monte_Carlo_integration) integration.

We multiply and divide by $q_\phi(z|x)$ to obtain a formulation of the integral as an _expected value_:

$$
\begin{align*}
p_\theta(x) &= \int q_\phi(z|x)\, \frac{p_\theta(x,z)}{q_\phi(z|x)}\, \textrm{d}z \\
&= \mathbb{E}_{z \sim q_\phi(z|x)} \frac{p_\theta(x,z)}{q_\phi(z|x)}\,.
\end{align*}
$$

On paper, this equivalence holds.  But our plan will be to approximate the expected value by drawing _samples_ and averaging.  For accuracy using some finite number of samples, it's important that $q_\phi(z|x)$ places [importance](https://en.wikipedia.org/wiki/Importance_sampling) in the right place, so that our samples are taken where the most probability mass is.

This effectively means that only a $q_\phi(z|x)$ that truly functions as an "encoder" -- placing high probability on latents $z$ that make the joint density $p_\theta(x,z)$ large -- will suffice.  Let's keep that in mind as we proceed.  If such a requirement falls out of the mathematics that follows, we'll know we're on the right track.

Taking logs,

$$
\log p_\theta(x) = \log \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \frac{p_\theta(x,z)}{q_\phi(z|x)}\right]\,.
$$

We'd like to take the log inside the expected value, but of course that's not a thing. There is [Jensen's inequality](https://en.wikipedia.org/wiki/Jensen%27s_inequality) though, so we can at least bound the result:

$$
\begin{align*}
\log p_\theta(x) &\geq \mathbb{E}_{z \sim q_\phi(z|x)}\left[ \log \frac{p_\theta(x,z)}{q_\phi(z|x)}\right] \\
&= \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \log p_\theta(x,z) - \log q_\phi(z|x)\right]\,.
\end{align*}
$$

So if we can maximise the right hand side, we are assured the left hand side, the log likelihood, will be larger still.  (There remains the question of how _tight_ the bound actually is -- see exercises.)

Expand the first term using $p_\theta(x,z) = p_\theta(x|z)\,p(z)$

$$
\log p_\theta(x) \geq \mathbb{E}_{z \sim q_\phi(z|x)} \left[ \log p_\theta(x|z) +  \log p(z) - \log q_\phi(z|x)\right]\,.\qquad (*)
$$

All three terms in this bound are under our control. The first and third terms involve $p_\theta(x|z)$ and $q_\phi(z|x)$ which are our _decoder_ and _encoder_ respectively. The second term is the PDF of our latent variable $z$, which is completely at our liberty to choose. Let's focus on that first.


## Latent space

Remember, we are building a mapping from image space $\mathcal{X}$ to latent space $\mathcal{Z}$, and we can dream up whatever latent space we like for $\mathcal{Z}$.  (It's then a matter of learning suitable encoding and decoding mappings between the spaces.)  So now it's time to make some concrete decisions about $\mathcal{Z}$.  We want $\mathcal{Z}$ to be as "nice" to work with as possible, so what could be nicer than choosing simply
$$
p(z) = \mathcal{N}(0,I_d).
$$

That is, $\mathcal{Z}$ is $\mathbb{R}^d$ and $p(z)$ is the standard normal distribution.  That's effectively fixing what kind of latent variables $z$ are plausible. With the choice of the standard normal distribution, we would expect all components to be between around $-3$ and $3$, most likely to be around zero, etc.  So it's just standard normal in $\mathbb{R}^d$, what could be simpler.

## Encoder

Let's do the encoder next, $q_\phi(z|x)$.  It takes in an image $x$, and returns a probability density function for the corresponding latent variable $z$.  Remember, we're _not_ mapping $x$ to a _deterministic_ latent $z$.  Instead, we're using the PDF as a way of capturing our _uncertainty_ in exactly where the image maps to in latent space.  

Again, the simplest distribution that allows us to capture this notion of uncertainty is a normal distribution, so why complicate matters any further?  We'll choose

$$
q_\phi(z|x) = \mathcal{N}(\mu_\phi(x),\  \Sigma_\phi(x))
$$

where the covariance matrix $\Sigma_\phi(x)$ is diagonal

$$
\Sigma_\phi(x) = \textrm{diag}(\sigma_\phi^2(x))
$$
and
$$
\mu_\phi(x),\ \sigma_\phi^2(x) \in \mathbb{R}^d\,.
$$

So each input image $x$ maps to a "Gaussian ellipsoid" centred on the mean $\mu_\phi(x) \in \mathcal{Z}$ with component-wise variance $\sigma_\phi^2(x)$.  Since $\sigma_\phi^2(x)$ is a vector, (i.e. $\Sigma$ is a diagonal covariance matrix) the model can have different per-component uncertainties.

All these lovely smooth normal distributions we're using ensures our latent space will be as pleasant to work with as it can be.

## Decoder

For the decoder, you might actually have expected there would be no uncertainty in mapping $z$ to $x$.  And that's nearly right. Refer back to the figure earlier, mapping $z$ to $x$ is a many-to-one situation; the opposite of the troublesome one-to-many situation for the encoder.  The only uncertainty in the decoding is really just the "thickness" of the manifold -- we've drawn the curve with quite a thick linestyle to emphasise this.  This is just acknowledging that any real image can be perturbed to some extent (i.e. change each pixel value by a small amount) and it would still be considered the same image.

So for the decoder, we map $z$ to a mean value $f_\theta(z) \in \mathcal{X}$, and use just a single scalar variance $\sigma^2$, not $z$-dependent, to represent the "thickness" of the manifold.

$$
p_\theta(x|z) = \mathcal{N}(f_\theta(z),\  \sigma^2 I)\,.
$$

# Putting it all together

We've now soaked up all the freedom we had in specifying distributions. With these choices, we can substitute into the bound $(*)$.

We have
$$
p_\theta(x|z) = (2\pi\sigma^2)^{-d/2}\,\exp\left(-\frac{\|x - f_\theta(z)\|^2}{2\sigma^2}\right)\,,
$$
$$
q_\phi(z|x) = (2\pi)^{-d/2}\textrm{det}(\Sigma_\phi(x))^{-1/2}\,\exp\left(-\frac{1}{2}(z-\mu_\phi(x))^\top\Sigma_\phi(x)^{-1} (z-\mu_\phi(x))\right)\,,
$$
$$
p(z) = (2\pi)^{-d/2}\,\exp\left(-\frac{\|z\|^2}{2}\right)
$$
where
$$
\Sigma_\phi(x) = \textrm{diag}(\sigma_\phi^2(x))\,.
$$


So we can take logs of all three expressions
$$
\log p_\theta(x|z) = -\frac{d}{2}\log(2\pi\sigma^2)\, -\frac{1}{2\sigma^2}  \|x - f_\theta(z)\|^2,
$$

$$
\begin{align*}
\log q_\phi(z|x) &= -\frac{d}{2}\log(2\pi) -\frac{1}{2} \log \det(\Sigma_\phi(x)) \, -\frac{1}{2}(z-\mu_\phi(x))^\top\Sigma_\phi(x)^{-1} (z-\mu_\phi(x))\\
&= -\frac{d}{2}\log(2\pi) -\frac{1}{2} \sum_{j=1}^d \log \sigma_{\phi,j}^2(x) \, -\frac{1}{2} \sum_{j=1}^d \frac{(z_j - \mu_{\phi,j}(x))^2}{\sigma_{\phi,j}^2(x)},
\end{align*}
$$

and

$$
\begin{align*}
\log p(z) &= -\frac{d}{2}\log(2\pi)\, -\frac{\|z\|^2}{2} \\
&= -\frac{d}{2}\log(2\pi)\, - \frac{1}{2}\sum_{j=1}^d z_j^2
\end{align*}
$$


And now we take expectations:
$$
\mathbb{E}_{z \sim q_\phi(z|x)} \log p_\theta(x|z) = -\frac{d}{2}\log(2\pi\sigma^2)\, -\frac{1}{2\sigma^2} \mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2
$$

$$
\mathbb{E}_{z \sim q_\phi(z|x)} \log q_\phi(z|x) = -\frac{d}{2}\log(2\pi)  -\frac{1}{2} \sum_{j=1}^d \log \sigma_{\phi,j}^2(x) \, - \frac{1}{2}\sum_{j=1}^d 1
$$

$$
\mathbb{E}_{z \sim q_\phi(z|x)} \log p(z) = -\frac{d}{2}\log(2\pi)\, - \frac{1}{2}\sum_{j=1}^d\left(\mu_{\phi,j}(x))^2 + \sigma_{\phi,j}^2(x)\right)
$$

## Loss function

So returning to $(*)$ and piecing together the three terms, we have

$$
\begin{align*}
\log p_\theta(x) &\geq \mathbb{E}_{z \sim q_\phi(z|x)} \log p_\theta(x|z) + \mathbb{E}_{z \sim q_\phi(z|x)} \log p(z) - \mathbb{E}_{z \sim q_\phi(z|x)} \log q_\phi(z|x) \\
&= -\frac{d}{2}\log(2\pi\sigma^2) -\frac{1}{2\sigma^2} \mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2 - \frac{1}{2} \sum_{j=1}^d \left( \mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1 \right)
\end{align*}
$$

Recalling where we started from, the log likelihood is given by

$$
S(\theta) = \frac{1}{N} \sum_{i=1}^N \log p_\theta(x_i) \approx \mathbb{E}_{x\sim p_\textrm{data}} [\log p_\theta(x)]\,.
$$

Substituting our bound above for $\log p_\theta(x)$ we can thus derive a suitable objective function.  As usual though, we prefer to minimise a loss function rather than maximising the objective function.  We also don't need to bother with the first term in the bound, since it's constant with respect to the parameters $\theta$ and $\phi$.  So dropping this term and flipping signs, we derive a _per-sample_ loss function

$$
\tilde{\ell}(\theta,\phi; x) = \frac{1}{2\sigma^2} \mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2 + \frac{1}{2} \sum_{j=1}^d \left( \mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1 \right)\,.
$$

One final step to tidy it up -- any scalar multiple of the loss function is as good as any other, so we can multiply through by the $2\sigma^2$ factor, label $\beta = \sigma^2$, and arrive at our final **per-sample loss function**

$$
\ell(\theta,\phi; x) = \mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2 + \beta \sum_{j=1}^d \left( \mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1 \right)\,.
$$

The total **loss function** over all samples is thus

$$
L(\theta, \phi) = \frac{1}{N} \sum_{i=1}^N \ell(\theta, \phi; x_i)\,.
$$

### Reconstruction error
Let's examine the first term of the per-sample loss $\ell(\theta, \phi; x)$:

$$
\mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2\,.
$$

This is the expected value of the round-trip reconstruction error for converting $x$ to $z \sim q_\phi(z|x)$ and back to $\hat{x} = f_\theta(z)$.  This enforces that the encoder and decoder do indeed function correctly as a pair, which we flagged earlier would be a sign that our approach was on the right track.  But how do we calculate this term?  The natural way is to sample some $z$s from the distribution $q_\phi(z|x)$ and compute the mean squared error.

So remembering
$$
q_\phi(z|x) = \mathcal{N}(\mu_\phi(x),\  \textrm{diag}(\sigma_\phi^2(x))
$$

a simple way to sample a $z$ from this distribution is to sample
$$
{\large \varepsilon}_k \sim \mathcal{N}(0,I)
$$

and compute
$$
z_k = \mu_\phi(x) + \sigma_\phi(x)\, \odot \, {\large \varepsilon}_k\,.
$$

Then we could average over $K$ such samples:
$$
\begin{align*}
\mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2 &\approx \frac{1}{K} \sum_{k=1}^K \| x - f_\theta(z_k) \| \\
&=  \frac{1}{K} \sum_{k=1}^K \| x - f_\theta(\mu_\phi(x) + \sigma_\phi(x)\, \odot \, {\large \varepsilon}_k) \|\,.
\end{align*}
$$


### Kullback-Leibler divergence
The second term in the per-sample loss is straightforward to calculate:

$$
\sum_{j=1}^d \left( \mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1 \right)\,,
$$

but what does it represent?  The clue is where it came from: the expression

$$
\mathbb{E}_{z \sim q_\phi(z|x)} \left[ \log q_\phi(z|x) - \log p(z)\right]\,.
$$

This quantity already has a name in the statistics literature: it's called the [Kullback-Leibler divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence) (**KL divergence**), denoted

$$
D_\textrm{KL}(q \| p) = \mathbb{E}_{z \sim q(z)} \left[ \log \frac{q(z)}{p(z)}  \right]\,.
$$

It measures how different the distribution $q$ is from the reference distribution $p$.  You can see how this plays out in the summation formula: the term

$$
\mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1
$$

will be small only if $\mu_{\phi,j}(x)\approx 0$ and $\sigma^2_{\phi,j}(x) \approx 1$.  So it acts to pull the latent distribution towards $\mathcal{N}(0,I)$.



# Summary

Let's summarise what we've done.  We started out with the standard idea of maximising the expected value of the log likelihood.  Through mathematical manipulation, we reformulated this into a minimisation problem for the loss function

$$
L(\theta, \phi) = \frac{1}{N} \sum_{i=1}^N \ell(\theta, \phi; x_i)
$$

where the per-sample loss function has the theoretically elegant form
$$
\ell(\theta,\phi; x) = \mathbb{E}_{z \sim q_\phi(z|x)}  \|x - f_\theta(z)\|^2 + \beta\, D_\textrm{KL}(q_\phi(z|x)\,\|\,p(z))
$$

with the practical formula for computing it

$$
\ell(\theta,\phi; x) \approx \frac{1}{K} \sum_{k=1}^K \| x - f_\theta(\mu_\phi(x) + \sigma_\phi(x)\, \odot \, {\large \varepsilon}_k) \| + \beta \sum_{j=1}^d \left( \mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1 \right)\,.
$$

Neither of these terms were chosen by us -- they fell out naturally from the mathematics.  But now that we see them, we recognise what they're doing.

The first term wants to ensure that the encoder and decoder work together to convert properly between $x$ and $z$.  You might have guessed that this term should be there from the outset, and perhaps even thought it should be the only relevant measure of success.

But no, the second term, the KL divergence term, has a say also.  Its job is to ensure the latent distribution does not stray far from the reference distribution $\mathcal{N}(0,I)$.  In fact, it actively penalises the encoder $q$ for any deviation from $\mathcal{N}(0,I)$.  So the encoder is not free to just map $x$ to any old $z$ in latent space.  A mapping that produced latent variables $z$ that were wildly inconsistent with a standard normal distribution would be heavily penalised by the KL divergence term.

(For further reading, this particular loss function has the terminology "[ELBO](https://en.wikipedia.org/wiki/Evidence_lower_bound)" associated with it, short for "Evidence lower bound".)

What if we only had the first term?  As described at the start of the lesson, we'd have an ordinary autoencoder, not a _variational_ autoencoder.  And experience shows that such encoders end up learning essentially lookup tables.  Each $x$ encodes to any old $z$ in latent space, and any such $z$ reliably decodes back to its corresponding $x$.  So the mean squared error term is minimised.  But the latent space is a mess.  There is no structure or continuity to it, the latent manifold is not simple to traverse, and you gain nothing by doing it beyond the simple compression in the number of coordinates needed to represent an image.

So it's the extra pull exerted by the KL divergence term that keeps things smooth and connected in latent space.  Notice though that we do have a lever to adjust the strength of this pull, through the parameter $\beta$.  Smaller $\beta$ will favour reconstruction accuracy over latent space smoothness, and larger $\beta$ will do the converse.  Remember that this trade-off also fell out of the mathematics naturally -- $\beta$ is not an ad-hoc "fudge factor".

# Encoder architecture

From this point onwards in our study, the neural network architectures will be getting more complex.  Still composed of the same familiar building blocks: dense layers, convolution layers, flattening, etc.  But with enough different pieces that it's not worth our time building them by hand.

Instead we will gain experience with building these networks using the deep learning framework, Lux in this case.  The following architecture is adapted from the [convolutional VAE example](https://github.com/LuxDL/Lux.jl/blob/main/examples/ConvolutionalVAE/main.jl). It comes as an encoder/decoder pair, each with their own learnable parameters.

The mathematical presentation to date has treated the input space as $\mathcal{X} = \mathbb{R}^{d_1}$, but the actual implementation very much exploits the 2D structure of images.  In particular, as the name suggests, the architectures are rich in convolutional layers.

To keep the computational complexity manageable on a single CPU, we will design the architecture assuming the training data will be images of dimension $32 \times 32$.

Here's the encoder first.

In [ ]:
function cvae_encoder(latent_dim)

    return @compact(
        embed = Chain(
            Conv((3, 3),  1 => 16; stride=2, pad=1), BatchNorm(16, swish),
            Conv((3, 3), 16 => 32; stride=2, pad=1), BatchNorm(32, swish),
            Conv((3, 3), 32 => 64; stride=2, pad=1), BatchNorm(64, swish),
            FlattenLayer()
        ),
        mu = Dense(1024 => latent_dim),
        logvar = Dense(1024 => latent_dim)
    ) do x
        y = embed(x)
        μ = mu(y)
        logσ² = clamp.(logvar(y), -20.0f0, 10.0f0)
        σ = exp.(logσ² .* 0.5f0)
        ϵ = randn_like(σ)
        z = μ .+ σ .* ϵ

        @return z, μ, logσ²
    end
end

#
The embedding step comprises multiple convolutional blocks. Each  block has a stride of 2, meaning it progressively downsamples the input spatially.  The image comes in as size $32 \times 32$, but length and width get halved with each block, ultimately down to a size of just $4 \times 4$.  But meanwhile each block _increases_ the number of _channels_, from 1 (greyscale input) to 16, 32 and 64.  The hope is that overall this portion of the network functions as a hierarchical feature extractor, with each layer being able to capture more, and higher-level, local patterns (e.g. edges to shapes to objects).

The batch norm layers are new to us.  During training, their job is to compute the batch mean and variance of each channel, and hence to normalise each layer of the batch, making them mean-centred and unit-variance. This acts to stabilise the activations.  It's particularly important for VAEs where we are training the encoder and decoder together, and hence relying on the the two separate networks to function well together as a pair.

A standard choice for the activation function for this application, which we have opted for, is [swish](https://en.wikipedia.org/wiki/Swish_function), defined by

$$
\textrm{swish}(t) = t\, \textrm{sigmoid}(t) = \frac{t}{1 + \textrm{e}^{-t}}\,.
$$

After these conv + batch norm layers, the data is of size $4 \times 4 \times 64$, and we just flatten the result to a 1024-dim vector, and run that through two separate dense layers to output the predicted mean $\mu_\phi(x)$ and variance $\sigma_\phi^2(x)$, which are of the latent dimension $d$.

Actually from the code you can see that we output $\log \sigma^2_\phi(x)$ rather than $\sigma_\phi^2(x)$.  There are two good reasons for this. First, we recall that $\log \sigma^2_\phi(x)$ actually appears in the loss function, so it's a value we'd require anyway. But second, outputting $\log \sigma^2_\phi(x)$ means we don't need to enforce any positivity constraint -- unlike $\sigma^2_\phi(x)$ which we'd have to somehow enforce to be positive.  You can also see there is a little extra stabilisation applied in the code to stop $\log \sigma^2$ getting too out of hand (which could cause $\sigma^2$ to over- or under-flow).

The final output is then, as we already derived
$$
z = \mu_\phi(x) + \sigma_\phi(x)\, \odot \, {\large \varepsilon}\
$$
and for use in the loss function the code also returns $\mu_\phi(x)$ and $\log \sigma^2_\phi(x)$.

# Decoder architecture

The decoder follows a somewhat similar pattern, albeit in reverse.  The latent input is expanded out to dimension 1024 by a dense layer, and then reshaped to size $4 \times 4 \times 64$. The conv layers now expand the image spatially progressively to $32 \times 32$, while reducing the number of channels back down to 1.  The final activation is `sigmoid` rather than `swish`, since that ensures the output is in the range $[0,1]$, which is what we need for an image.

In [ ]:
function cvae_decoder(latent_dim)
    return @compact(

        seed = Dense(latent_dim => 1024),

        upchain = Chain(
            Upsample(2), Conv((3, 3), 64 => 32; stride = 1, pad = 1), BatchNorm(32, swish),
            Upsample(2), Conv((3, 3), 32 => 16; stride = 1, pad = 1), BatchNorm(16, swish),
            Upsample(2), Conv((3, 3), 16 => 1, sigmoid; stride = 1, pad = 1)
        )
    ) do z
        y = seed(z)
        img = reshape(y, 4, 4, 64, :)
        @return upchain(img)
    end
end

# Parameters

These two networks have their own trainable parameters, which we denoted $\phi$ and $\theta$ respectively in the mathematical derivation. But we don't learn them separately: the encoder and decoder are trained as a pair.  So it's natural to keep the two components together in a simple structure which we'll call `CVAE`.

In [ ]:
struct CVAE <: AbstractLuxContainerLayer{(:encoder, :decoder)}
    encoder
    decoder
end

function CVAE(latent_dim)
    encoder = cvae_encoder(latent_dim)
    decoder = cvae_decoder(latent_dim)
    return CVAE(encoder, decoder)
end

# Model instantiation

For today we will opt for a latent dimension of only 2, with the aim to build a visual intuition of what's happening.

In [ ]:
latent_dim = 2
cvae = CVAE(latent_dim)

# Parameter initialisation

In total we have about 50k parameters, which means this model is trainable on a CPU, provided we don't throw too much training data at it. We initialise the parameters as usual.

We've kept with our convention of denoting the parameters by `p`, but in fact we have two sets of parameters now, `p.encoder`, corresponding to $\phi$, and `p.decoder`, corresponding to $\theta$. We also have, for the first time, a nonempty _state_ `st`.

You can see amongst the output there are arrays of `running_mean` and `running_var`, initialised to 0 and 1 respectively.  These are for the benefit of the batch norm layers.  We'll talk more about them later, but for now just keep in mind that our neural networks are actually stateful.

In [ ]:
rng = Random.default_rng()
p0, st0 = Lux.setup(rng, cvae);

st0.encoder  # similarly st0.decoder

# MNIST training data

For training data we will return to our old friend, the MNIST training set, where each image has been pre-padded to size $32 \times 32$ to match our encoder's expected input size.

For this worksheet we will just train on a subset of 10,000 images.

In [ ]:
MNISTfile = jldopen(download("https://github.com/moroneyt/MXB301/raw/main/resources/MNIST32.jld2"))
images = MNISTfile["images"] / 255f0  # convert to [0,1]
labels = MNISTfile["labels"]

nsamples = 10_000
batchsize = 100
data_loader = DataLoader(images[:,:,:,1:nsamples]; batchsize, shuffle=true)

# Loss function

Now for the loss function.  Recall there are two components to the per-sample loss: the reconstruction error and the KL regularisation.

$$
\ell(\theta,\phi; x) \approx \frac{1}{K} \sum_{k=1}^K \| x - f_\theta(\mu_\phi(x) + \sigma_\phi(x)\, \odot \, {\large \varepsilon}_k) \| + \beta \sum_{j=1}^d \left( \mu_{\phi,j}(x)^2 + \sigma^2_{\phi,j}(x) - \log \sigma_{\phi,j}^2(x) - 1 \right)\,.
$$

Here's the implementation.  Notably, it uses the value $K=1$ -- so a _single sample_ is used in the first term to approximate the expected reconstruction error.  This is obviously quite a noisy approximation, but given that we will use stochastic gradient descent to train anyway, it turns out that $K=1$ works fine in practice, and it's obviously the cheapest value to use.

In [ ]:
function loss(cvae, p, st, x)
    mseloss = MSELoss(; agg=sum)  # summed, rather than averaged

    # Encode x to z
    (z, μ, logσ²), st_enc = cvae.encoder(x, p.encoder, st.encoder)

    # Decode z to x̂
    x̂, st_dec = cvae.decoder(z, p.decoder, st.decoder)

    # We approximate the expectation with a single term!
    reconstruction_loss = mseloss(x̂, x)

    # KL regularisation term
    kldiv_loss = sum(μ.^2 .+ exp.(logσ²) .- logσ² .- 1)

    # Total loss is weighted combination of both
    loss = reconstruction_loss + β * kldiv_loss

    # Also return the updated state and anything else that might be useful
    state = (; encoder=st_enc, decoder=st_dec)
    extras = (; μ, logσ²)

    return loss, state, extras
end

# Optimiser

We'll stick with the usual Adam optimiser, with default parameters.  Initialise the trainer to keep track of all the bookkeeping.

In [ ]:
optimiser = Lux.Optimisers.Adam()
trainer = Training.TrainState(cvae, p0, st0, optimiser)

# Regularisation parameter
Before we commence training, we need to decide on a value for the regularisation parameter $\beta$.  As you remember, this controls the relative importance of the two terms in the loss function: the reconstruction error (favoured for small $\beta$) and the KL divergence regularisation (favoured for large $\beta$).  In the absence of any analysis whatsoever, we will simply choose $\beta = 1$.

In [ ]:
β = 1.0f0

# Training
Now we can train.  For this worksheet we'll just take 10 epochs, but you can adjust this if you have the patience to wait longer (you can also increase the number of training samples).  Expect it to take around one minute per epoch.

In [ ]:
autodiff = AutoZygote()  # reverse mode AD
epochs = 10

@time for epoch = 1:epochs
  loss_total = 0.0f0
  for X in data_loader

    # Compute loss and gradient
    g, lossval, extras, trainer = Training.compute_gradients(autodiff, loss, X, trainer)

    # Take gradient step and adapt learning rate
    trainer = Training.apply_gradients(trainer, g)

    # Accumulate loss
    loss_total += lossval
   end

   @show train_loss = loss_total / length(data_loader)
  end

p = trainer.parameters;
st = trainer.states;

# Running means and variances

Now that it's trained, we want to use the network for inference.  That is, we want to start encoding some images, and see where they end up in latent space.  But we appear to have a problem: the batch norm layers.  You recall the batch norm layer's job is to compute the batch mean and variance of each channel, and hence to normalise each layer of the batch, making them mean-centred and unit-variance.  But in inference, we don't have a batch!  We might only be encoding a single image.  This is why we have the state variable `st`, to keep hold of a _running mean_ and variance for each batch norm layer.  Here, you can see all the running means and variances now recorded, after having gone through training.

In [ ]:
st.encoder  # also st.decoder

# Test mode
These running means and variances are used at inference time, in place of actually computing a batch mean and variance.  This way, the images we're feeding through the network at inference time undergo a comparable mean-centring to the data that was used at training time.

All this means, when we define our usual helper functions for inference, we should turn on _test mode_ so that the network knows to apply the running means and variances saved in the state, rather than computing batch means and variances.

In [ ]:
# Helper functions for inference
encode(x) = cvae.encoder(x, p.encoder, Lux.testmode(st.encoder))[1] # test mode
decode(z) = cvae.decoder(z, p.decoder, Lux.testmode(st.decoder))[1] # test mode

# Inference
Well what are waiting for?  Let's encode all of our training images to see where they have ended up in latent space!

The array `z` contains the $10,000$ 2-dimensional latent vectors. If the KL divergence term has done its job, these latent coordinates should have a mean of around $0$ and variance around $1$.

The output suggests this is indeed the case.

In [ ]:
z, μ, _ = encode(images[:,:,:,1:nsamples])
@show summary(z)
mean(z; dims=2), var(z; dims=2)

# Visualisation
Now let's visualise where each digit actually ended up in latent space.

We see that indeed, altogether the points are consistent with being drawn from a standard normal distribution in 2D.  But take a closer look, and there is structure to be found. Most obviously, there is strong clustering of digits by their label.  We didn't tell it to do this!  The digit label is nowhere in the training data nor in the loss function.

In [ ]:
function plot_latent_space(z)

    fig = Figure()
    ax = Axis(fig[1,1], limits=(-3,3,-3,3), aspect=1, title="Latent space")

    ndigitpoints = 500  # how many dots for each digit
    for digit in 0:9
        idx = findall(labels .== digit)
        cols = ColorSchemes.tab10.colors  # vector of 10 colours
        c = cols[digit+1]

        scatter!(ax,
            z[1, idx[1:ndigitpoints]],
            z[2, idx[1:ndigitpoints]],
            color = c,
            markersize = 6,
            label = string(digit)
        )

    end

    Legend(fig[1,2], ax)
    fig

end

fig = plot_latent_space(z)

#
Rather, this natural clustering reflects the two pressures the VAE feels in its loss function.  It must map images to latents that it can reliably reconstruct from.  But $p(z)$ must not stray far from $\mathcal{N}(0,I)$. The model is therefore forced to learn the most efficient representation of digits in latent space, so that it has room to encode all the information it requires without needing to send digits to far-flung regions of latent space.

The mechanism by which this is understood to occur is that the convolutional layers learn about edges, strokes, curves, thickness, etc.  And in terms of these higher-level features, every digit `0` is very much like every other digit `0`, but not very much like a digit `1`, etc.  In the last stage of the encoder, it is this semantically-rich feature representation that gets turned into a latent variable.

So there is structure in the encoding, but just enough so that overall the distribution of all latent variables still resembles $\mathcal{N}(0,I)$.

#
This is even clearer if we compare side-by-side with a true draw of the same number of points from $\mathcal{N}(0,I)$.

This extra structure -- the degree of departure from $\mathcal{N}(0,I)$ -- is how the network has learned to encode the information it needed to reconstruct images from these latents.  But the loss function, with its KL divergence regularisation, ensures the model can deviate from $\mathcal{N}(0,I)$ only when the pay-off for doing so is sufficiently improved reconstruction accuracy.  So the two terms work in harmony to make the model as economical as possible in the amount of information it actually uses to build latent representations, thereby keeping the lovely smooth structure of latent space that we desire.

In [ ]:
scatter(fig[1,3], randn(5000), randn(5000), markersize=6,
    axis=(limits=(-3,3,-3,3), aspect=1, title = L"\mathcal{N}(0,I)"))
fig

#
On that last point, you could imagine tracing a closed, continuous path through latent space and decoding back to an image with each step.  Because the latent representation is continuous, this should generate a continuous morphing of one digit into another in image space.  Below is just such an animation we prepared earlier.

<img src="https://github.com/moroneyt/MXB301/raw/main/resources/morphing_digits.gif">

Notice how the decoder is always mapping to something that looks at least "digit-like", even if it's not a true digit.  Again this is because the latent representation is based on the features learned by the convolutional layers.  So when decoding from a latent vector, it's mapping onto the most plausible combination of those features.  And those features always combine to give a "digit-looking thing" -- blurry yes, but never just pixel noise.

# Conclusion

In this lesson we learned:

* how autoencoders compress high-dimensional data into a lower-dimensional latent representation

* how probability is used as a way of representing uncertainty in a CVAE

* why and how the loss function incorporates reconstruction accuracy and the Kullback-Leibler divergence term

* how to build a simple CVAE using a deep learning library

* how to train the model on the MNIST data set and interpret the results

A CVAE is certainly a powerful tool for semantically meaningful image compression.  But it is not satisfactory as a generative model.  You cannot simply choose a random $z$ from $\mathcal{N}(0,I)$ and decode it to generate new digits.  Sometimes that will work, as we saw above, but other times you'll get only a "digit-looking thing" which is somewhere between two actual digits.

For more complex distributions of images: cats, forests, buildings, whatever, it's clearly unsatisfactory for a generative model to return an interpolation between images;  some sort of hybrid Eiffel Tower / Taj Mahal, or whatever.

Instead, for a true generative model, we require a procedure for starting from an $\mathcal{N}(0,I)$ sample and dynamically evolving the position until it lands somewhere in a region of latent space that corresponds to real images.  Only then can you decode, and be assured of generating a plausible new image.

On to the next lesson!